# Prerequisites: PyTorch and Computer Vision Basics

The notebooks in this folder assume a handful of ideas that the Coursera course never spells out, because the original was written in TensorFlow. This notebook covers them, with runnable examples you can poke at.

**What's here**

1. [How an image is represented](#1) — layout, dtype, and the channels-first convention
2. [What each transform actually does](#2) — seen side by side on one photo
3. [Normalization, and how to undo it for display](#3)
4. [Shape surgery: `unsqueeze`, `squeeze`, `permute`, `reshape`](#4)
5. [`Dataset` and `DataLoader`](#5)
6. [Convolution arithmetic](#6) — including the transposed convolutions used by FCN-8 and U-Net
7. [Logits, probabilities and the matching losses](#7)
8. [Anatomy of a training loop](#8)
9. [Exploring the datasets used in this project](#9)

Nothing here is graded. Run it top to bottom, or jump to the section you need.

In [ ]:
import os
import glob

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)
print("torch", torch.__version__)

DATA = "data"
torch.manual_seed(0)
np.random.seed(0)

<a name="1"></a>
## 1. How an image is represented

This is the single biggest difference from the TensorFlow notebooks.

| | TensorFlow / Keras / PIL / OpenCV | PyTorch |
| --- | --- | --- |
| Layout | **channels last**, `(height, width, channels)` | **channels first**, `(channels, height, width)` |
| Batched | `(N, H, W, C)` | `(N, C, H, W)` |
| Typical dtype | `uint8` in `[0, 255]`, or `float32` in `[0, 1]` | `float32` in `[0, 1]` after `ToTensor()` |

Every time you move an image between matplotlib (which wants channels last) and a PyTorch model (which wants channels first), you need a `permute`. Getting this wrong is the most common source of confusing errors in these labs.

In [ ]:
def sample_image_path():
    '''
    Finds an image to use for the demonstrations in this notebook.

    Prefers one of the cat/dog pictures the Week 4 labs download; otherwise falls back
    to any JPEG under the data folder.

    Returns:
      string or None -- path to an image, or None if nothing suitable was found
    '''
    for candidate in ["data/dog1.jpg", "data/cat1.jpg", "dog1.jpg", "cat1.jpg"]:
        if os.path.exists(candidate):
            return candidate
    found = glob.glob(os.path.join(DATA, "**", "*.jpg"), recursive=True)
    return found[0] if found else None


path = sample_image_path()
print("using:", path)

if path is None:
    print("No image found. Run any Week 4 lab once to download the sample pictures.")
else:
    pil_image = Image.open(path).convert("RGB")
    print("PIL size (width, height):", pil_image.size)

    as_numpy = np.array(pil_image)
    print("numpy shape (H, W, C):   ", as_numpy.shape, as_numpy.dtype, "range", as_numpy.min(), "to", as_numpy.max())

    as_tensor = transforms.functional.to_tensor(pil_image)
    print("tensor shape (C, H, W):  ", tuple(as_tensor.shape), as_tensor.dtype,
          "range", round(float(as_tensor.min()), 3), "to", round(float(as_tensor.max()), 3))

Notice what `to_tensor` did in one step: it moved the channel axis to the front **and** divided by 255 to put the values in `[0, 1]`. In Keras those were two separate concerns (`rescale=1./255` on the generator, and no axis move at all).

To show a tensor with matplotlib you have to undo the axis move:

In [ ]:
def show_tensor(image_tensor, title="", ax=None):
    '''
    Displays a channels-first image tensor with matplotlib.

    Args:
      image_tensor (tensor) -- image of shape (3, H, W) or (1, H, W), values in [0, 1]
      title (string) -- title drawn above the image
      ax (matplotlib axis) -- axis to draw on, or None to use the current one

    Returns:
      matplotlib axis -- the axis the image was drawn on
    '''
    ax = ax or plt.gca()
    array = image_tensor.detach().cpu()
    if array.shape[0] == 1:                 # grayscale
        ax.imshow(array.squeeze(0), cmap="gray")
    else:                                   # (C, H, W) -> (H, W, C) for matplotlib
        ax.imshow(array.permute(1, 2, 0).clamp(0, 1))
    ax.set_title(title, fontsize=10)
    ax.axis("off")
    return ax


if path is not None:
    plt.figure(figsize=(4, 4))
    show_tensor(as_tensor, "permute(1, 2, 0) before imshow")
    plt.show()

<a name="2"></a>
## 2. What each transform actually does

`torchvision.transforms` replaces Keras' `ImageDataGenerator`. The mapping used across these labs is:

| Keras `ImageDataGenerator` | torchvision transform |
| --- | --- |
| `rescale=1./255` | `ToTensor()` (also moves the channel axis) |
| `rotation_range=40` | `RandomAffine(degrees=40)` |
| `width_shift_range=0.2, height_shift_range=0.2` | `RandomAffine(translate=(0.2, 0.2))` |
| `shear_range=0.2` | `RandomAffine(shear=0.2)` |
| `zoom_range=0.2` | `RandomAffine(scale=(0.8, 1.2))` |
| `horizontal_flip=True` | `RandomHorizontalFlip()` |
| `target_size=(150, 150)` | `Resize((150, 150))` |

Two things worth internalising:

- **Order matters.** Geometric transforms operate on PIL images, `ToTensor` converts, and `Normalize` operates on tensors. So the tensor conversion sits between them.
- **Random transforms re-roll every time the item is fetched.** That is why a training `Dataset` produces a slightly different image each epoch, and why the validation pipeline must not include them.

Let's see each one on its own.

In [ ]:
def compare_transforms(image, named_transforms, ncols=4):
    '''
    Applies each transform to the same image and plots the results in a grid.

    Args:
      image (PIL.Image) -- the picture every transform is applied to
      named_transforms (list) -- (name, transform) pairs, each transform returning a tensor
      ncols (int) -- how many images per row

    Returns:
      None -- the grid is drawn with matplotlib
    '''
    n = len(named_transforms)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 3.4 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for ax, (name, transform) in zip(axes, named_transforms):
        show_tensor(transform(image), name, ax=ax)
    for ax in axes[n:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


if path is not None:
    to_tensor = transforms.ToTensor()
    small = transforms.Resize((150, 150))

    demos = [
        ("original (resized 150)",      transforms.Compose([small, to_tensor])),
        ("RandomHorizontalFlip(p=1)",   transforms.Compose([small, transforms.RandomHorizontalFlip(p=1.0), to_tensor])),
        ("RandomAffine(degrees=40)",    transforms.Compose([small, transforms.RandomAffine(degrees=40), to_tensor])),
        ("translate=(0.2, 0.2)",        transforms.Compose([small, transforms.RandomAffine(0, translate=(0.2, 0.2)), to_tensor])),
        ("shear=20",                    transforms.Compose([small, transforms.RandomAffine(0, shear=20), to_tensor])),
        ("scale=(0.6, 0.6)  (zoom out)",transforms.Compose([small, transforms.RandomAffine(0, scale=(0.6, 0.6)), to_tensor])),
        ("CenterCrop(80)",              transforms.Compose([small, transforms.CenterCrop(80), to_tensor])),
        ("ColorJitter(brightness=.8)",  transforms.Compose([small, transforms.ColorJitter(brightness=0.8), to_tensor])),
        ("Grayscale(3 channels)",       transforms.Compose([small, transforms.Grayscale(3), to_tensor])),
        ("Resize((40, 150)) squashed",  transforms.Compose([transforms.Resize((40, 150)), to_tensor])),
    ]
    compare_transforms(pil_image, demos)

Run the cell again and the random ones will look different, while `original` and `CenterCrop` stay put. That is the whole idea of augmentation: the model never sees exactly the same training image twice.

**A trap worth knowing**: resizing a *segmentation mask* must use nearest-neighbour interpolation. Any smoothing would average neighbouring label ids and invent classes that do not exist. Compare:

In [ ]:
def resize_mask_demo():
    '''
    Shows why segmentation masks must be resized with nearest-neighbour interpolation.

    Builds a tiny label map with ids 0, 1 and 2, enlarges it both ways, and reports
    which label values survive.

    Returns:
      None -- results are printed and plotted
    '''
    mask = np.zeros((8, 8), dtype=np.uint8)
    mask[2:6, 2:6] = 1
    mask[3:5, 3:5] = 2
    mask_image = Image.fromarray(mask)

    nearest = np.array(mask_image.resize((64, 64), Image.NEAREST))
    bilinear = np.array(mask_image.resize((64, 64), Image.BILINEAR))

    print("original label ids :", sorted(np.unique(mask).tolist()))
    print("after NEAREST      :", sorted(np.unique(nearest).tolist()), " <- unchanged, correct")
    print("after BILINEAR     :", sorted(np.unique(bilinear).tolist()), " <- invented ids, wrong")

    fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
    for ax, (img, name) in zip(axes, [(mask, "original 8x8"), (nearest, "NEAREST"), (bilinear, "BILINEAR")]):
        ax.imshow(img, cmap="viridis", vmin=0, vmax=2)
        ax.set_title(name, fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


resize_mask_demo()

<a name="3"></a>
## 3. Normalization, and how to undo it for display

Pretrained torchvision weights were trained on inputs normalized with the ImageNet channel statistics:

```
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]
```

`Normalize` subtracts the mean and divides by the standard deviation, per channel. After it, pixel values are roughly centred on zero and are **no longer in `[0, 1]`**, so passing the result straight to `imshow` gives you the washed out, over-saturated picture you may have seen. You have to invert it first.

Two labs in this folder use a different convention, and it is worth knowing why:

- **InceptionV3** (the cats vs dogs transfer lab) expects `[-1, 1]`, which is `Normalize([0.5]*3, [0.5]*3)`.
- The **GradCAM and Saliency labs** wrap `Normalize` *inside* the model, so the image you hand in stays in `[0, 1]`. That matters because the saliency map is the gradient with respect to whatever you feed in, and you want it aligned with pixels you can actually look at.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def denormalize(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    '''
    Reverses a Normalize step so an image can be displayed.

    Args:
      tensor (tensor) -- normalized image, shape (3, H, W)
      mean (list of float) -- the per-channel mean that was subtracted
      std (list of float) -- the per-channel standard deviation that was divided out

    Returns:
      tensor -- the image back in the [0, 1] range, shape (3, H, W)
    '''
    mean_t = torch.tensor(mean).view(3, 1, 1)
    std_t = torch.tensor(std).view(3, 1, 1)
    return tensor.detach().cpu() * std_t + mean_t


if path is not None:
    plain = transforms.Compose([transforms.Resize((150, 150)), transforms.ToTensor()])(pil_image)
    normed = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)(plain)

    print(f"before Normalize: min {plain.min():+.3f}  max {plain.max():+.3f}  mean {plain.mean():+.3f}")
    print(f"after  Normalize: min {normed.min():+.3f}  max {normed.max():+.3f}  mean {normed.mean():+.3f}")

    fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))
    show_tensor(plain, "[0, 1] as loaded", ax=axes[0])
    show_tensor(normed, "normalized, shown raw\n(clipped, looks wrong)", ax=axes[1])
    show_tensor(denormalize(normed), "normalized then denormalized", ax=axes[2])
    plt.tight_layout()
    plt.show()

<a name="4"></a>
## 4. Shape surgery

Four operations account for nearly every shape change in these notebooks.

| Operation | What it does | Typical use here |
| --- | --- | --- |
| `unsqueeze(d)` | inserts a size-1 axis at position `d` | adding the batch axis: `(3, H, W)` to `(1, 3, H, W)` |
| `squeeze(d)` | removes size-1 axes | dropping the batch or channel axis before plotting |
| `permute(...)` | reorders axes | channels-first to channels-last for matplotlib |
| `reshape` / `view` | reinterprets the same data with new dimensions | flattening features before a `Linear` layer |

`permute` and `reshape` are not interchangeable. `permute` moves axes and keeps each pixel's identity; `reshape` reinterprets the buffer and will silently scramble an image if you use it to swap axes.

In [ ]:
x = torch.arange(2 * 3 * 4).reshape(2, 3, 4)
print("start                       ", tuple(x.shape))
print("unsqueeze(0)   -> batch axis", tuple(x.unsqueeze(0).shape))
print("unsqueeze(1)               ", tuple(x.unsqueeze(1).shape))
print("permute(1, 2, 0)           ", tuple(x.permute(1, 2, 0).shape))
print("reshape(2, 12)             ", tuple(x.reshape(2, 12).shape))
print("flatten(1)                 ", tuple(x.flatten(1).shape))

y = torch.zeros(1, 1, 5, 5)
print()
print("squeeze() removes every size-1 axis:", tuple(y.squeeze().shape))
print("squeeze(0) removes only axis 0:     ", tuple(y.squeeze(0).shape))

print()
print("permute vs reshape are NOT the same:")
img = torch.arange(12).reshape(3, 2, 2)                 # (C, H, W)
print("  permute(1,2,0)[0,0] =", img.permute(1, 2, 0)[0, 0].tolist(), " <- the 3 channels of pixel (0,0), correct")
print("  reshape(2,2,3)[0,0] =", img.reshape(2, 2, 3)[0, 0].tolist(), " <- unrelated numbers, wrong")

<a name="5"></a>
## 5. `Dataset` and `DataLoader`

These two replace the whole `tf.data` pipeline.

- A **`Dataset`** answers two questions: how many items are there (`__len__`), and give me item `i` (`__getitem__`). It deals with **one** example at a time, and that is where preprocessing lives.
- A **`DataLoader`** wraps a `Dataset` and handles batching, shuffling, and loading in background worker processes.

The equivalences:

| `tf.data` | PyTorch |
| --- | --- |
| `dataset.map(fn)` | do the work inside `__getitem__` |
| `.batch(32)` | `DataLoader(..., batch_size=32)` |
| `.shuffle(1024)` | `DataLoader(..., shuffle=True)` |
| `.prefetch(AUTOTUNE)` | `DataLoader(..., num_workers=4)` |
| `.repeat()` | not needed; just loop over the loader again |

In [ ]:
class SquaresDataset(Dataset):
    '''A tiny dataset that returns a coloured square and its colour label, to show the mechanics.'''

    def __init__(self, n, size=32, transform=None):
        '''
        Stores how many squares to generate and how to preprocess them.

        Args:
          n (int) -- how many items the dataset holds
          size (int) -- height and width of each generated square
          transform (callable) -- optional transform applied to the PIL image
        '''
        self.n = n
        self.size = size
        self.transform = transform

    def __len__(self):
        '''
        Reports how many items the dataset holds.

        Returns:
          int -- number of items in the dataset
        '''
        return self.n

    def __getitem__(self, idx):
        '''
        Builds item `idx` on demand, which is where per-example preprocessing belongs.

        Args:
          idx (int) -- index of the item to build

        Returns:
          (tensor, int) -- the image and its class label
        '''
        label = idx % 3
        colour = [(220, 60, 60), (60, 180, 90), (70, 110, 230)][label]
        image = Image.new("RGB", (self.size, self.size), colour)
        if self.transform is not None:
            image = self.transform(image)
        return image, label


demo = SquaresDataset(12, transform=transforms.ToTensor())
print("len(dataset)        :", len(demo))
one_image, one_label = demo[0]
print("dataset[0] image    :", tuple(one_image.shape), "label:", one_label, "  <- ONE example, no batch axis")

loader = DataLoader(demo, batch_size=4, shuffle=False)
batch_images, batch_labels = next(iter(loader))
print("one batch images    :", tuple(batch_images.shape), "  <- batch axis added by the DataLoader")
print("one batch labels    :", tuple(batch_labels.shape), batch_labels.tolist())
print("batches per epoch   :", len(loader))

**A macOS specific gotcha** that appears in several notebooks here. `DataLoader(num_workers>0)` starts worker processes. On macOS the default start method is `spawn`, and a spawned worker re-imports your module. A `Dataset` class defined *inside a notebook* does not exist in that fresh process, so the workers crash. Passing `multiprocessing_context="fork"` fixes it, which is why you see this near the top of several labs:

```python
MP_CONTEXT = "fork" if platform.system() == "Darwin" else None
```

<a name="6"></a>
## 6. Convolution arithmetic

You need this to read the model definitions, and especially to understand the decoders in the segmentation labs.

For a convolution or pooling layer:

$$H_{out} = \left\lfloor \frac{H_{in} + 2 \times padding - kernel}{stride} \right\rfloor + 1$$

`padding='same'` (with stride 1) chooses the padding that keeps the size unchanged, which is what Keras' `padding='same'` did.

For a **transposed** convolution, which the FCN-8 and U-Net decoders use to upsample, the formula runs the other way:

$$H_{out} = (H_{in} - 1) \times stride - 2 \times padding + kernel + output\_padding$$

In [ ]:
def layer_shapes(layers, input_shape=(1, 3, 224, 224)):
    '''
    Runs a dummy tensor through each layer and reports how the shape changes.

    Args:
      layers (list) -- (name, module) pairs applied in order
      input_shape (tuple) -- shape of the dummy input tensor

    Returns:
      None -- a table is printed
    '''
    x = torch.zeros(input_shape)
    print(f"  {'layer':38s} {'output shape':>22s}")
    print(f"  {'input':38s} {str(tuple(x.shape)):>22s}")
    for name, layer in layers:
        x = layer(x)
        print(f"  {name:38s} {str(tuple(x.shape)):>22s}")


print("Downsampling, the VGG-16 encoder pattern:")
layer_shapes([
    ("Conv2d(3, 64, 3, padding='same')",  nn.Conv2d(3, 64, 3, padding='same')),
    ("MaxPool2d(2)",                       nn.MaxPool2d(2)),
    ("Conv2d(64, 128, 3, padding='same')", nn.Conv2d(64, 128, 3, padding='same')),
    ("MaxPool2d(2)",                       nn.MaxPool2d(2)),
    ("AdaptiveAvgPool2d(1)",               nn.AdaptiveAvgPool2d(1)),
    ("Flatten()",                          nn.Flatten()),
])

In [ ]:
print("Upsampling, the U-Net decoder pattern. These two settings exactly double the size:")
layer_shapes([
    ("ConvTranspose2d(64,32,3,s=2,p=1,op=1)", nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1)),
    ("ConvTranspose2d(32,16,3,s=2,p=1,op=1)", nn.ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1)),
], input_shape=(1, 64, 16, 16))

print()
print("The FCN-8 pattern instead overshoots and crops, which is why you see o[:, :, 1:-1, 1:-1]:")
x = torch.zeros(1, 12, 7, 7)
up = nn.ConvTranspose2d(12, 12, kernel_size=4, stride=2, bias=False)(x)
print("  ConvTranspose2d(k=4, s=2) on 7x7 ->", tuple(up.shape), "(overshoots to 16)")
print("  after cropping [1:-1, 1:-1]      ->", tuple(up[:, :, 1:-1, 1:-1].shape), "(matches the 14x14 skip connection)")

<a name="7"></a>
## 7. Logits, probabilities, and the matching loss

Keras models usually ended in `activation='softmax'` or `'sigmoid'`, and the loss consumed probabilities. **PyTorch models conventionally return raw scores, called logits**, and the loss applies the activation internally. This is numerically more stable, and it is the convention every notebook in this folder follows.

| Task | Final layer | Loss | Target it expects |
| --- | --- | --- | --- |
| Multi-class | `Linear(n, num_classes)`, no activation | `nn.CrossEntropyLoss` | class **indices**, shape `(N,)`, dtype long |
| Binary | `Linear(n, 1)`, no activation | `nn.BCEWithLogitsLoss` | floats 0.0 or 1.0, shape `(N, 1)` |
| Per-pixel segmentation | `Conv2d(n, num_classes, 1)` | `nn.CrossEntropyLoss` | label map, shape `(N, H, W)`, dtype long |

The binary row explains the line you will see in the cats vs dogs lab:

```python
yb = yb.float().unsqueeze(1)   # (N,) long -> (N, 1) float, to match the logits
```

In [ ]:
logits = torch.tensor([[2.0, 0.5, -1.0]])
print("logits              :", logits.tolist())
print("softmax             :", torch.softmax(logits, dim=1).round(decimals=3).tolist())
print("argmax is identical either way:", int(logits.argmax(1)), int(torch.softmax(logits, 1).argmax(1)))
print()

targets = torch.tensor([0])
manual = -torch.log(torch.softmax(logits, 1))[0, targets[0]]
print(f"CrossEntropyLoss on logits      : {nn.CrossEntropyLoss()(logits, targets):.6f}")
print(f"computed by hand from softmax   : {manual:.6f}   <- same value, so the softmax is built in")
print()

binary_logits = torch.tensor([[1.5], [-2.0]])
binary_targets = torch.tensor([[1.0], [0.0]])
print(f"BCEWithLogitsLoss               : {nn.BCEWithLogitsLoss()(binary_logits, binary_targets):.6f}")
print("sigmoid(logit) > 0.5 is the same test as logit > 0, which is why the labs write `logits > 0`")

<a name="8"></a>
## 8. Anatomy of a training loop

`model.fit()` does not exist. Every notebook here writes the loop out, and it is always the same five steps.

In [ ]:
def train_demo(model, loader, loss_fn, optimizer, device, epochs=3):
    '''
    A minimal but complete training loop, showing the five steps every lab repeats.

    Args:
      model (nn.Module) -- the network being trained
      loader (DataLoader) -- yields (inputs, targets) batches
      loss_fn (callable) -- loss applied to (logits, targets)
      optimizer (Optimizer) -- updates the weights
      device (torch.device) -- device the batches are moved to
      epochs (int) -- how many passes over the loader

    Returns:
      list of float -- the mean loss of each epoch
    '''
    losses = []
    for epoch in range(epochs):
        model.train()                                     # 1. training mode: dropout/batchnorm behave accordingly
        running, count = 0.0, 0
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)

            logits = model(inputs)                        # 2. forward pass
            loss = loss_fn(logits, targets)

            optimizer.zero_grad()                         # 3. clear gradients left from the previous step
            loss.backward()                               # 4. backward pass, fills .grad on every parameter
            optimizer.step()                              # 5. update the weights

            running += loss.item() * len(inputs)
            count += len(inputs)
        losses.append(running / count)
        print(f"  epoch {epoch + 1}: loss {losses[-1]:.4f}")
    return losses


tiny_model = nn.Sequential(nn.Flatten(), nn.Linear(3 * 32 * 32, 3)).to(device)
tiny_loader = DataLoader(SquaresDataset(60, transform=transforms.ToTensor()), batch_size=10, shuffle=True)

print("training a throwaway model on the coloured squares:")
train_demo(tiny_model, tiny_loader, nn.CrossEntropyLoss(),
           torch.optim.Adam(tiny_model.parameters(), lr=1e-3), device, epochs=3)

Two details that bite people:

- **`optimizer.zero_grad()` is not optional.** PyTorch *accumulates* gradients into `.grad`. Skip the zeroing and every step uses the sum of all previous gradients.
- **`model.train()` and `model.eval()` matter** whenever the network has dropout or batch normalization. In eval mode dropout is disabled and batch norm uses its running statistics. Wrapping evaluation in `torch.no_grad()` additionally stops autograd from recording, which saves memory and time.

<a name="9"></a>
## 9. Exploring the datasets used in this project

Each section below is guarded, so it only runs if you have already downloaded that dataset by running the matching lab. Nothing here downloads anything new.

In [ ]:
def describe_array(name, array):
    '''
    Prints the shape, dtype and value range of an array.

    Args:
      name (string) -- label shown at the start of the line
      array (array) -- the array to describe

    Returns:
      None -- a single line is printed
    '''
    array = np.asarray(array)
    print(f"  {name:22s} shape {str(array.shape):22s} dtype {str(array.dtype):9s} "
          f"range {array.min()} to {array.max()}")


def show_grid(images, titles=None, ncols=8, cmap=None, suptitle=""):
    '''
    Plots a row or grid of images.

    Args:
      images (list) -- images as (H, W) or (H, W, 3) arrays
      titles (list of str) -- optional per-image titles
      ncols (int) -- images per row
      cmap (string) -- matplotlib colormap, useful for grayscale and label maps
      suptitle (string) -- title for the whole figure

    Returns:
      None -- the grid is drawn with matplotlib
    '''
    n = len(images)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(1.7 * ncols, 2.0 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for i, ax in enumerate(axes):
        if i < n:
            ax.imshow(images[i], cmap=cmap)
            if titles is not None:
                ax.set_title(str(titles[i]), fontsize=8)
        ax.axis("off")
    if suptitle:
        fig.suptitle(suptitle, fontsize=11)
    plt.tight_layout()
    plt.show()

### 9.1 Fashion MNIST and MNIST — the Week 1 and Week 4 labs

In [ ]:
from torchvision.datasets import FashionMNIST, MNIST

for cls, name in [(FashionMNIST, "Fashion MNIST"), (MNIST, "MNIST")]:
    folder = os.path.join(DATA, cls.__name__, "raw")
    if not os.path.exists(folder):
        print(f"{name}: not downloaded yet, run the matching lab first")
        continue

    ds = cls(root=DATA, train=True, download=False)
    print(f"{name}: {len(ds)} training images, {len(ds.classes)} classes")
    describe_array("images", ds.data.numpy())
    describe_array("labels", ds.targets.numpy())

    counts = np.bincount(ds.targets.numpy())
    print("  class balance:", "perfectly balanced" if counts.std() < 1 else f"min {counts.min()}, max {counts.max()}")

    idx = np.random.choice(len(ds), 8, replace=False)
    show_grid([ds.data[i].numpy() for i in idx],
              [ds.classes[ds.targets[i]].split("/")[0] for i in idx],
              cmap="gray", suptitle=f"{name} samples")

### 9.2 CIFAR-10 — the transfer learning lab

In [ ]:
from torchvision.datasets import CIFAR10

if os.path.exists(os.path.join(DATA, "cifar-10-batches-py")):
    ds = CIFAR10(root=DATA, train=True, download=False)
    print(f"CIFAR-10: {len(ds)} training images, {len(ds.classes)} classes")
    describe_array("images", ds.data)
    counts = np.bincount(np.array(ds.targets))
    print("  per class:", dict(zip(ds.classes, counts.tolist())))
    print("  note: 32x32 is tiny. The lab upsamples 7x to 224x224 so ResNet50's pretrained")
    print("        filters, which were learned at 224x224, are operating at a familiar scale.")

    idx = np.random.choice(len(ds), 8, replace=False)
    show_grid([ds.data[i] for i in idx], [ds.classes[ds.targets[i]] for i in idx],
              suptitle="CIFAR-10 samples")
else:
    print("CIFAR-10 not downloaded yet, run C3_W1_Lab_2 first")

### 9.3 Cats vs Dogs — the Week 1 and Week 4 labs

In [ ]:
pet_root = os.path.join(DATA, "catsdogs", "PetImages")

def survey_cats_vs_dogs(root, sample=400):
    '''
    Reports how many images each class has, how many files are unreadable, and the size spread.

    Args:
      root (string) -- folder containing the Cat and Dog subfolders
      sample (int) -- how many files per class to open when measuring sizes

    Returns:
      None -- findings are printed and a size scatter is plotted
    '''
    sizes, bad = [], 0
    for label in ["Cat", "Dog"]:
        files = sorted(os.listdir(os.path.join(root, label)))
        print(f"  {label}: {len(files)} files")
        for name in files[:sample]:
            try:
                with Image.open(os.path.join(root, label, name)) as im:
                    im.verify()
                with Image.open(os.path.join(root, label, name)) as im:
                    sizes.append(im.size)
            except Exception:
                bad += 1

    print(f"  unreadable in the {2 * sample} sampled files: {bad}")
    print("  (the full dataset contains a few zero-byte and non-image files, which is why the")
    print("   labs filter with is_valid_image before building the split)")

    widths, heights = zip(*sizes)
    print(f"  width  {min(widths)} to {max(widths)}, median {int(np.median(widths))}")
    print(f"  height {min(heights)} to {max(heights)}, median {int(np.median(heights))}")
    print("  sizes vary a lot, which is why every pipeline starts with a Resize")

    plt.figure(figsize=(5, 4))
    plt.scatter(widths, heights, s=4, alpha=0.3)
    plt.xlabel("width"); plt.ylabel("height"); plt.title("Cats vs Dogs image sizes")
    plt.tight_layout(); plt.show()


if os.path.exists(pet_root):
    survey_cats_vs_dogs(pet_root)
else:
    print("Cats vs Dogs not downloaded yet, run C3_W1_Lab_1 first")

### 9.4 CamVid — the FCN-8 segmentation lab

In [ ]:
camvid = os.path.join(DATA, "fcnn", "dataset1")

if os.path.exists(camvid):
    class_names = ['sky', 'building', 'column/pole', 'road', 'side walk', 'vegetation',
                   'traffic light', 'fence', 'vehicle', 'pedestrian', 'byciclist', 'void']
    images = sorted(glob.glob(os.path.join(camvid, "images_prepped_train", "*.png")))
    labels = sorted(glob.glob(os.path.join(camvid, "annotations_prepped_train", "*.png")))
    print(f"CamVid: {len(images)} train images, {len(labels)} label maps")

    sample = np.array(Image.open(images[0]))
    mask = np.array(Image.open(labels[0]))
    describe_array("image", sample)
    describe_array("label map", mask)
    print("  the label map is a single channel of class ids, NOT a colour picture")

    pixel_counts = np.zeros(12, dtype=np.int64)
    for f in labels[:60]:
        pixel_counts += np.bincount(np.array(Image.open(f)).ravel(), minlength=12)[:12]
    share = 100 * pixel_counts / pixel_counts.sum()

    print("\n  pixel share per class (60 images):")
    for name, pct in sorted(zip(class_names, share), key=lambda t: -t[1]):
        print(f"    {name:14s} {pct:5.1f}%")
    print("  heavily imbalanced. Sky, building and road dominate, so a model that predicts")
    print("  only those three still scores a high pixel accuracy. That is exactly why the")
    print("  lab reports per-class IOU and Dice instead of accuracy alone.")

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
    axes[0].imshow(sample); axes[0].set_title("frame"); axes[0].axis("off")
    axes[1].imshow(mask, cmap="tab20", vmin=0, vmax=11); axes[1].set_title("label map"); axes[1].axis("off")
    plt.tight_layout(); plt.show()
else:
    print("CamVid not downloaded yet, run C3_W3_Lab_1 first")

### 9.5 M2NIST — the Week 3 segmentation assignment

In [ ]:
m2nist = os.path.join(DATA, "m2nist")

if os.path.exists(os.path.join(m2nist, "combined.npy")):
    images = np.load(os.path.join(m2nist, "combined.npy"), mmap_mode="r")
    segments = np.load(os.path.join(m2nist, "segmented.npy"), mmap_mode="r")
    print(f"M2NIST: {images.shape[0]} samples")
    describe_array("images", images[:200])
    describe_array("masks (one-hot)", segments[:50])
    print("  the masks ship one-hot over 11 channels: digits 0 to 9 plus background at index 10.")
    print("  CrossEntropyLoss wants class ids instead, which is why the assignment argmaxes them.")

    label_maps = np.argmax(segments[:200], axis=-1)
    counts = np.bincount(label_maps.ravel(), minlength=11)
    share = 100 * counts / counts.sum()
    print(f"\n  background covers {share[10]:.1f}% of all pixels; each digit averages {share[:10].mean():.2f}%")

    show_grid([images[i] for i in range(6)] + [label_maps[i] for i in range(6)],
              ["image"] * 6 + ["labels"] * 6, ncols=6, cmap="gray",
              suptitle="M2NIST: images (top) and their label maps (bottom)")
else:
    print("M2NIST not downloaded yet, run C3W3_Assignment first")

### 9.6 Caltech Birds — the Week 1 bounding box assignment

In [ ]:
birds = os.path.join(DATA, "caltech_birds2010", "0.1.1")

def survey_birds(folder, limit=300):
    '''
    Reports image count and bounding box statistics for the Caltech Birds shards.

    Args:
      folder (string) -- directory holding the TFRecord shards
      limit (int) -- how many records to read

    Returns:
      None -- findings are printed and a histogram is drawn
    '''
    from tfrecord.reader import tfrecord_loader
    shard = sorted(glob.glob(os.path.join(folder, "*train*")))[0]

    areas, widths = [], []
    for i, record in enumerate(tfrecord_loader(shard, None, description={"image": "byte", "bbox": "float"})):
        if i >= limit:
            break
        ymin, xmin, ymax, xmax = record["bbox"]
        areas.append(float((ymax - ymin) * (xmax - xmin)))
        widths.append(float(xmax - xmin))

    print(f"  read {len(areas)} records from one shard")
    print("  boxes are stored normalized as (ymin, xmin, ymax, xmax), values in [0, 1].")
    print("  the assignment reorders them to (xmin, ymin, xmax, ymax), which is easy to get wrong.")
    print(f"  box covers {100 * np.mean(areas):.0f}% of the image on average "
          f"(min {100 * min(areas):.0f}%, max {100 * max(areas):.0f}%)")
    print("  the bird usually fills much of the frame, so even a mediocre model scores a fair IoU")

    plt.figure(figsize=(5, 3.4))
    plt.hist(areas, bins=30)
    plt.xlabel("box area as a fraction of the image"); plt.ylabel("count")
    plt.title("Caltech Birds box sizes"); plt.tight_layout(); plt.show()


if os.path.exists(birds):
    survey_birds(birds)
else:
    print("Caltech Birds not downloaded yet, run C3W1_Assignment first")

### 9.7 Oxford-IIIT Pets — the U-Net lab

In [ ]:
pets = os.path.join(DATA, "oxford-iiit-pet")

if os.path.exists(os.path.join(pets, "annotations", "trimaps")):
    masks = sorted(glob.glob(os.path.join(pets, "annotations", "trimaps", "*.png")))
    images = sorted(glob.glob(os.path.join(pets, "images", "*.jpg")))
    print(f"Oxford-IIIT Pets: {len(images)} images, {len(masks)} masks")

    raw = np.array(Image.open(masks[0]))
    describe_array("raw mask", raw)
    print("  raw ids are 1 = foreground, 2 = background, 3 = not classified.")
    print("  the lab subtracts 1 to get 0 = pet, 1 = background, 2 = outline.")

    counts = np.zeros(4, dtype=np.int64)
    for f in masks[:80]:
        counts += np.bincount(np.array(Image.open(f)).ravel(), minlength=4)[:4]
    share = 100 * counts / counts.sum()
    for raw_id, name in [(1, "pet"), (2, "background"), (3, "outline")]:
        print(f"    {name:12s} {share[raw_id]:5.1f}%")
    print("  the outline class is thin and rare, so it is the hardest of the three to learn")

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
    axes[0].imshow(Image.open(images[0]).convert("RGB")); axes[0].set_title("image"); axes[0].axis("off")
    axes[1].imshow(raw, cmap="viridis"); axes[1].set_title("trimap"); axes[1].axis("off")
    plt.tight_layout(); plt.show()
else:
    print("Oxford-IIIT Pets not downloaded yet, run C3_W3_Lab_2 first")

<a name="10"></a>
## 10. Why is each step necessary?

Every notebook in this folder repeats the same handful of operations: resize, `ToTensor`, `Normalize`, `shuffle=True`, `zero_grad()`, `model.eval()`, `torch.no_grad()`, raw logits. It is easy to copy them without knowing what they buy you.

Each task below **removes one of them and shows what breaks**. Try to predict the outcome before running the cell. They all run in seconds.

### Task 1: Why resize every image to the same size?

The cats and dogs photos range from roughly 100 to 500 pixels on a side. What happens if you skip the `Resize` and hand a batch of mixed sizes to a `DataLoader`?

**Predict:** does it work, silently pad, or fail?

In [ ]:
class MixedSizeDataset(Dataset):
    '''Returns images of deliberately different sizes, to show why Resize is mandatory.'''

    def __init__(self, sizes, transform=None):
        '''
        Stores the side lengths to generate and the transform applied to each image.

        Args:
          sizes (list of int) -- the side length of each generated square image
          transform (callable) -- optional transform applied to each PIL image

        Returns:
          None -- this is a constructor
        '''
        self.sizes = sizes
        self.transform = transform

    def __len__(self):
        '''
        Reports how many images the dataset holds.

        Returns:
          int -- number of images in the dataset
        '''
        return len(self.sizes)

    def __getitem__(self, idx):
        '''
        Builds image `idx` at its assigned size.

        Args:
          idx (int) -- index of the image to build

        Returns:
          (tensor, int) -- the image and a dummy label
        '''
        size = self.sizes[idx]
        image = Image.new("RGB", (size, size), (128, 128, 128))
        if self.transform is not None:
            image = self.transform(image)
        return image, 0


def why_resize():
    '''
    Demonstrates that a DataLoader cannot stack images of different sizes into a batch.

    Returns:
      None -- the failure and the fix are printed
    '''
    sizes = [64, 96, 128, 80]

    print("WITHOUT Resize:")
    loader = DataLoader(MixedSizeDataset(sizes, transforms.ToTensor()), batch_size=4)
    try:
        images, _ = next(iter(loader))
        print("   batched to", tuple(images.shape))
    except RuntimeError as e:
        print("   RuntimeError:", str(e).split("\n")[0][:110])
        print("   a batch is ONE tensor of shape (N, C, H, W), so every image must share H and W")

    print("\nWITH Resize((100, 100)):")
    fixed = transforms.Compose([transforms.Resize((100, 100)), transforms.ToTensor()])
    images, _ = next(iter(DataLoader(MixedSizeDataset(sizes, fixed), batch_size=4)))
    print("   batched to", tuple(images.shape), " <- works")


why_resize()

### Task 2: Why `ToTensor` instead of just `np.array(image)`?

`ToTensor` does two things at once. Skip it and you hit both problems in turn.

**Predict:** what does `Conv2d` complain about if you feed it a channels-last uint8 array?

In [ ]:
def why_to_tensor(image):
    '''
    Shows the two jobs ToTensor does: moving the channel axis, and scaling to [0, 1].

    Args:
      image (PIL.Image) -- any RGB picture

    Returns:
      None -- each failure and its fix are printed
    '''
    conv = nn.Conv2d(3, 8, 3)
    raw = torch.from_numpy(np.array(image.resize((64, 64))))          # (H, W, C) uint8
    print("raw numpy array   :", tuple(raw.shape), raw.dtype)

    print("\n1. channel axis in the wrong place:")
    try:
        conv(raw.unsqueeze(0).float())
    except RuntimeError as e:
        print("   RuntimeError:", str(e).split("\n")[0][:110])
        print("   Conv2d reads axis 1 as channels. Here axis 1 is height, so it sees 64 channels.")

    print("\n2. fixed by permute, but the values are still 0 to 255:")
    permuted = raw.permute(2, 0, 1).unsqueeze(0).float()
    out_big = conv(permuted)
    out_small = conv(permuted / 255.0)
    print(f"   activations from 0-255 input : std {out_big.std():8.3f}")
    print(f"   activations from 0-1   input : std {out_small.std():8.3f}")
    print(f"   {out_big.std() / out_small.std():.0f}x larger. Those feed into the loss and then the")
    print("   gradients, so an unscaled input makes the very first optimizer step enormous.")

    print("\nToTensor does the permute AND the divide by 255 in one call:")
    print("   ", tuple(transforms.ToTensor()(image.resize((64, 64))).shape))


if path is not None:
    why_to_tensor(pil_image)

### Task 3: Why normalize with the ImageNet statistics?

Pretrained weights were fitted to inputs with a particular mean and spread. Feeding them `[0, 1]` data instead shifts every activation, and the effect compounds through the network.

**Predict:** does the wrong normalization change the model's answer, or only its confidence?

In [ ]:
def why_normalize(image):
    '''
    Compares a pretrained classifier's predictions with and without ImageNet normalization.

    Args:
      image (PIL.Image) -- any RGB picture

    Returns:
      None -- both predictions are printed side by side
    '''
    from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

    weights = MobileNet_V2_Weights.IMAGENET1K_V1
    model = mobilenet_v2(weights=weights).eval()
    names = weights.meta["categories"]

    plain = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])(image).unsqueeze(0)
    normed = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)(plain)

    for label, batch in [("WITHOUT Normalize", plain), ("WITH Normalize", normed)]:
        with torch.no_grad():
            probs = torch.softmax(model(batch), dim=1)[0]
        top = probs.topk(3)
        guesses = ", ".join(f"{names[i]} {float(v):.2f}" for v, i in zip(top.values, top.indices))
        print(f"  {label:18s} {guesses}")

    print("\n  The un-normalized input is brighter and flatter than anything the network saw in")
    print("  training, so its answer is both different and far less certain. The weights were")
    print("  fitted to a specific input distribution; you have to reproduce it.")


if path is not None:
    why_normalize(pil_image)

### Task 4: Why `shuffle=True`?

The cats and dogs folder is read in order: every cat, then every dog. With `shuffle=False`, each batch contains exactly one class.

**Predict:** can the model still learn, or does it fail outright?

In [ ]:
def why_shuffle(device, epochs=6):
    '''
    Trains the same model on sorted and on shuffled batches, and compares the outcome.

    Args:
      device (torch.device) -- device the model runs on
      epochs (int) -- how many passes over the data

    Returns:
      dict -- the per-batch loss spread and the final accuracy for each setting
    '''
    generator = torch.Generator().manual_seed(1)
    features = torch.cat([torch.randn(150, 4, generator=generator) - 1.2,
                          torch.randn(150, 4, generator=generator) + 1.2])
    labels = torch.cat([torch.zeros(150), torch.ones(150)]).long()   # sorted: all 0s, then all 1s
    data = torch.utils.data.TensorDataset(features, labels)

    results = {}
    for name, shuffle in [("shuffle=False (sorted)", False), ("shuffle=True", True)]:
        torch.manual_seed(0)
        model = nn.Sequential(nn.Linear(4, 16), nn.BatchNorm1d(16), nn.ReLU(), nn.Linear(16, 2)).to(device)
        optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
        loader = DataLoader(data, batch_size=30, shuffle=shuffle)

        batch_losses = []
        for _ in range(epochs):
            model.train()
            for xb, yb in loader:
                xb, yb = xb.to(device), yb.to(device)
                loss = nn.CrossEntropyLoss()(model(xb), yb)
                optimizer.zero_grad(); loss.backward(); optimizer.step()
                batch_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            accuracy = float((model(features.to(device)).argmax(1) == labels.to(device)).float().mean())
        results[name] = (float(np.std(batch_losses)), accuracy)
        print(f"  {name:24s} per-batch loss swing {np.std(batch_losses):.3f}   final accuracy {accuracy:.3f}")

    print()
    print("  0.500 is coin-flip accuracy on a two-class problem, so the sorted run learned nothing.")
    print("  Every batch holds a single class, so the gradient points at that class alone, and")
    print("  batch normalization computes its statistics from one class too, meaning the running")
    print("  averages it stores for evaluation are never right. Shuffling fixes both at once.")
    return results


why_shuffle(device)

### Task 5: Why `optimizer.zero_grad()`?

PyTorch *accumulates* into `.grad` rather than replacing it. This is deliberate, since it lets you split a large batch across several backward passes. But it means forgetting to clear is a silent bug, not an error.

**Predict:** what are the three printed gradients?

In [ ]:
def why_zero_grad():
    '''
    Shows that gradients accumulate across backward passes unless explicitly cleared.

    Returns:
      None -- the accumulating and the cleared gradients are printed
    '''
    print("WITHOUT zero_grad, the same computation three times:")
    w = torch.ones(1, requires_grad=True)
    for step in range(1, 4):
        (w * 3).sum().backward()
        print(f"   step {step}: w.grad = {w.grad.item():.0f}")
    print("   the true gradient is 3 every time, but it is being summed: 3, 6, 9")

    print("\nWITH zero_grad:")
    w = torch.ones(1, requires_grad=True)
    for step in range(1, 4):
        if w.grad is not None:
            w.grad = None
        (w * 3).sum().backward()
        print(f"   step {step}: w.grad = {w.grad.item():.0f}")
    print("   correct. Skipping this makes every step larger than the last, and training diverges")
    print("   for no visible reason, which is why it appears in every loop in this folder.")


why_zero_grad()

### Task 6: Why `model.eval()` and `torch.no_grad()`?

They do two unrelated things, and you almost always want both when evaluating.

**Predict:** does calling `model.eval()` on a network with dropout change its output?

In [ ]:
def why_eval_and_no_grad(device):
    '''
    Shows that eval mode changes dropout and batch norm, and that no_grad stops graph recording.

    Args:
      device (torch.device) -- device the modules run on

    Returns:
      None -- the differences are printed
    '''
    torch.manual_seed(0)

    dropout = nn.Dropout(0.5)
    x = torch.randn(2, 8)

    dropout.train()
    print("nn.Dropout in train mode:")
    print(f"   two runs on the SAME input differ: {not torch.allclose(dropout(x), dropout(x))}")
    print("   a random half of the activations is zeroed on each call")

    dropout.eval()
    print("nn.Dropout in eval mode:")
    print(f"   two runs on the same input differ: {not torch.allclose(dropout(x), dropout(x))}")
    print("   dropout is disabled, so predictions are repeatable")

    batch_norm = nn.BatchNorm1d(4)
    batch = torch.randn(6, 4) * 3 + 5
    batch_norm.train(); trained = batch_norm(batch)
    batch_norm.eval();  evaluated = batch_norm(batch)
    print()
    print("nn.BatchNorm1d on the very same batch:")
    print(f"   train and eval outputs differ: {not torch.allclose(trained, evaluated)}")
    print("   train mode normalizes using THIS batch's statistics, eval mode uses the running")
    print("   averages. Evaluate in train mode and an image's score depends on which other")
    print("   images happened to share its batch, which is not a prediction you can trust.")

    print()
    print("torch.no_grad() is a separate concern. It controls whether autograd records:")
    model = nn.Linear(8, 2).to(device)
    xb = torch.randn(2, 8).to(device)
    with torch.no_grad():
        without = model(xb)
    with_grad = model(xb)
    print(f"   requires_grad inside  no_grad : {without.requires_grad}")
    print(f"   requires_grad outside no_grad : {with_grad.requires_grad}")
    print("   recording keeps every intermediate tensor alive for a backward pass you are not")
    print("   going to make. Skip it in every evaluation and prediction cell.")


why_eval_and_no_grad(device)

### Task 7: Why do the models return logits instead of probabilities?

Every model in this folder ends in a bare `Linear` or `Conv2d` with no softmax. The Keras originals ended in `activation='softmax'`. This is not a style preference.

**Predict:** what is the loss when the model is confidently wrong, computed each way?

In [ ]:
def why_logits():
    '''
    Shows that computing cross entropy from probabilities underflows to infinity.

    Returns:
      None -- both computations are printed for comparison
    '''
    logits = torch.tensor([[200.0, -200.0]])     # extremely confident, and wrong
    target = torch.tensor([1])

    probs = torch.softmax(logits, dim=1)
    print("logits              :", logits.tolist()[0])
    print("softmax             :", probs.tolist()[0], " <- the true class is exactly 0.0")

    manual = -torch.log(probs[0, target[0]])
    stable = F.cross_entropy(logits, target)
    print()
    print(f"  -log(probability)          : {manual.item()}    <- gradient is meaningless")
    print(f"  CrossEntropyLoss on logits : {stable.item():.1f}   <- finite and correct")
    print()
    print("  exp(-400) underflows to zero in float32, so the softmax throws away the very")
    print("  information the loss needs. log_softmax keeps it by working in log space the")
    print("  whole way. Letting the loss do both steps together is why the models in this")
    print("  folder hand it raw scores instead of probabilities.")


why_logits()

### Task 8: Why per-class IOU instead of accuracy, on segmentation?

This one matters for reading your own results. In M2NIST the background covers most of every image.

**Predict:** what pixel accuracy does a model score if it predicts "background" for every single pixel?

In [ ]:
def why_per_class_metrics(data_dir="data"):
    '''
    Scores a do-nothing baseline that labels every pixel background, on the real M2NIST masks.

    Args:
      data_dir (string) -- folder holding the m2nist arrays

    Returns:
      None -- accuracy and per-class IOU of the trivial baseline are printed
    '''
    seg_path = os.path.join(data_dir, "m2nist", "segmented.npy")
    if not os.path.exists(seg_path):
        print("M2NIST not downloaded yet, run C3W3_Assignment first")
        return

    truth = np.argmax(np.load(seg_path, mmap_mode="r")[:300], axis=-1)   # (300, 64, 84) class ids
    prediction = np.full_like(truth, 10)                                 # 10 is the background class

    accuracy = (prediction == truth).mean()
    print(f"  a model that ALWAYS predicts background scores {100 * accuracy:.1f}% pixel accuracy")

    print("\n  per-class IOU for that same do-nothing model:")
    for i in [0, 1, 2, 10]:
        intersection = np.sum((prediction == i) & (truth == i))
        union = np.sum((prediction == i) | (truth == i))
        name = "background" if i == 10 else f"digit {i}"
        print(f"    {name:12s} {intersection / max(union, 1):.3f}")

    print("\n  Over 90% accurate and completely useless. Accuracy is dominated by the majority")
    print("  class, so it cannot tell a working segmenter from a broken one. Per-class IOU")
    print("  drops to zero for every digit, which is why the labs grade on IOU and Dice.")


why_per_class_metrics()

### Task 9: Why augment the training data?

Augmentation is the one step whose benefit is invisible on the training set, which makes it easy to drop.

**Predict:** which run has the smaller gap between training and validation accuracy?

In [ ]:
def why_augmentation(device, n_train=64, epochs=12):
    '''
    Trains the same small model on a tiny dataset with and without augmentation.

    Args:
      device (torch.device) -- device the model runs on
      n_train (int) -- how many training images to use, kept small so overfitting shows
      epochs (int) -- passes over the training set

    Returns:
      None -- the train and validation accuracy of both runs are printed
    '''
    def make_shape(idx, rng):
        '''
        Draws a square or a circle at a random position, as a two-class toy problem.

        Args:
          idx (int) -- index of the item; even indices give squares, odd ones circles
          rng (Generator) -- numpy random generator that picks the position

        Returns:
          (PIL.Image, int) -- the 32x32 grayscale picture and its class label
        '''
        image = Image.new("L", (32, 32), 0)
        from PIL import ImageDraw
        draw = ImageDraw.Draw(image)
        x, y = rng.integers(4, 16), rng.integers(4, 16)
        if idx % 2 == 0:
            draw.rectangle([x, y, x + 12, y + 12], fill=255)
        else:
            draw.ellipse([x, y, x + 12, y + 12], fill=255)
        return image, idx % 2

    def build(n, seed):
        '''
        Builds a list of (PIL image, label) pairs with a fixed seed.

        Args:
          n (int) -- how many items to generate
          seed (int) -- seed for the position generator, so runs are reproducible

        Returns:
          list -- (PIL.Image, int) pairs
        '''
        rng = np.random.default_rng(seed)
        return [make_shape(i, rng) for i in range(n)]

    train_raw, val_raw = build(n_train, 0), build(200, 99)
    plain = transforms.ToTensor()
    augment = transforms.Compose([transforms.RandomAffine(degrees=25, translate=(0.25, 0.25)),
                                 transforms.ToTensor()])

    def run(train_transform, label):
        '''
        Trains one model and reports its train and validation accuracy.

        Args:
          train_transform (callable) -- transform applied to each training image
          label (string) -- name printed alongside the results

        Returns:
          None -- the two accuracies and their gap are printed
        '''
        torch.manual_seed(0)
        model = nn.Sequential(nn.Flatten(), nn.Linear(32 * 32, 32), nn.ReLU(), nn.Linear(32, 2)).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

        for _ in range(epochs):
            model.train()
            for image, y in train_raw:
                xb = train_transform(image).unsqueeze(0).to(device)
                yb = torch.tensor([y]).to(device)
                loss = nn.CrossEntropyLoss()(model(xb), yb)
                optimizer.zero_grad(); loss.backward(); optimizer.step()

        model.eval()
        def score(pairs):
            '''
            Measures accuracy on a list of (image, label) pairs, without augmentation.

            Args:
              pairs (list) -- (PIL.Image, int) pairs to evaluate

            Returns:
              float -- fraction classified correctly
            '''
            with torch.no_grad():
                correct = sum(int(model(plain(im).unsqueeze(0).to(device)).argmax()) == y for im, y in pairs)
            return correct / len(pairs)

        tr, va = score(train_raw), score(val_raw)
        print(f"  {label:22s} train {tr:.3f}   validation {va:.3f}   gap {tr - va:+.3f}")

    run(plain, "no augmentation")
    run(augment, "with augmentation")
    print("\n  Without augmentation the model memorises where each shape sat in the 64 training")
    print("  images and does worse on unseen positions. Augmentation shows it the same shape")
    print("  shifted and rotated, so it has to learn the shape instead of the position.")


why_augmentation(device)

### Recap

| You always see | Because without it |
| --- | --- |
| `Resize((H, W))` | a batch is one tensor, so mixed sizes cannot be stacked at all |
| `ToTensor()` | `Conv2d` reads the wrong axis as channels, and 0-255 inputs make the first step enormous |
| `Normalize(mean, std)` | pretrained weights see a distribution they were never fitted to, and get less confident and less accurate |
| `shuffle=True` | each batch holds one class, so gradients lurch and the model never settles |
| `optimizer.zero_grad()` | gradients silently accumulate and every step grows |
| `model.eval()` | dropout keeps firing and batch norm keeps using batch statistics, so predictions are not repeatable |
| `torch.no_grad()` | autograd records a graph you never use, wasting memory and time |
| raw logits, not softmax | confident predictions underflow and the loss becomes infinity |
| per-class IOU, not accuracy | a model predicting only background scores over 90% and looks fine |
| augmentation | the model memorises positions instead of learning shapes |

## Where to go next

With those pieces in place, the rest of the folder should read cleanly:

- **Week 1** applies transfer learning, and adds a bounding box regression head. Sections 1 to 3 and 7 are the relevant background.
- **Week 2** uses pretrained detectors and fine-tunes one. Section 1 matters most, since detection APIs are picky about box format and ordering.
- **Week 3** builds FCN-8 and U-Net segmentation models. Section 6 covers the transposed convolutions and the cropping, and section 9 shows why per-class metrics beat accuracy on these datasets.
- **Week 4** visualizes what a model looks at, using CAM, Grad-CAM and saliency. Sections 3 and 4 are what make those notebooks readable, since they are mostly shape manipulation and careful normalization.